In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [7]:
# ============================================================
# PROJECT PATH CONFIGURATION
# ============================================================

from pathlib import Path

# Notebook is running from:
# enterprise_hr_ai/notebooks

CURRENT_DIR = Path.cwd()

# Project root
PROJECT_ROOT = CURRENT_DIR.parent

# Raw data is stored in project_root/data/raw
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Processed data should also be stored in project_root/data/processed
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

# Create processed folder if it doesn't exist
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

print("Current Working Directory:")
print(CURRENT_DIR)

print("\nProject Root:")
print(PROJECT_ROOT)

print("\nRaw Data Path:")
print(RAW_DATA_PATH)

print("\nRaw Data Exists:", RAW_DATA_PATH.exists())

print("\nProcessed Data Path:")
print(PROCESSED_DATA_PATH)

print("\nProcessed Data Exists:", PROCESSED_DATA_PATH.exists())

Current Working Directory:
C:\Users\Shubham\Desktop\enterprise_hr_ai\notebooks

Project Root:
C:\Users\Shubham\Desktop\enterprise_hr_ai

Raw Data Path:
C:\Users\Shubham\Desktop\enterprise_hr_ai\data\raw

Raw Data Exists: True

Processed Data Path:
C:\Users\Shubham\Desktop\enterprise_hr_ai\data\processed

Processed Data Exists: True


In [8]:
# ============================================================
# LOAD RAW DATASETS
# ============================================================

datasets = {
    "employee_attrition": RAW_DATA_PATH / "employee_attrition.csv",
    "performance_engagement": RAW_DATA_PATH / "hr_performance_engagement.csv",
    "occupational_data": RAW_DATA_PATH / "occupation_data.csv",
    "essential_skills": RAW_DATA_PATH / "essential_skills.csv",
    "software_skills": RAW_DATA_PATH / "software_skills.csv"
    
}

raw_data = {}

for name, path in datasets.items():
    try:
        raw_data[name] = pd.read_csv(path)
        print(f"✓ Loaded {name}: {raw_data[name].shape}")
    except Exception as e:
        print(f"✗ Error loading {name}: {e}")

✓ Loaded employee_attrition: (1470, 35)
✓ Loaded performance_engagement: (5000, 13)
✓ Loaded occupational_data: (1016, 3)
✓ Loaded essential_skills: (18200, 15)
✓ Loaded software_skills: (31821, 7)


In [9]:
# ============================================================
# DATA CLEANING PROFILE - BEFORE CLEANING
# ============================================================

cleaning_profile = []

for name, df in raw_data.items():
    
    cleaning_profile.append({
        "Dataset": name,
        "Rows Before": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values Before": int(df.isnull().sum().sum()),
        "Duplicate Rows Before": int(df.duplicated().sum())
    })

cleaning_profile_df = pd.DataFrame(cleaning_profile)

cleaning_profile_df

,Dataset,Rows Before,Columns,Missing Values Before,Duplicate Rows Before
0,employee_attrition,1470,35,0,0
1,performance_engagement,5000,13,0,0
2,occupational_data,1016,3,0,0
3,essential_skills,18200,15,9100,0
4,software_skills,31821,7,0,0


In [21]:
# ============================================================
# DETAILED MISSING VALUE ANALYSIS
# ============================================================

missing_value_details = []

for dataset_name, df in raw_data.items():
    
    for column in df.columns:
        
        missing_count = df[column].isnull().sum()
        
        if missing_count > 0:
            
            missing_percentage = (
                missing_count / len(df)
            ) * 100
            
            missing_value_details.append({
                "Dataset": dataset_name,
                "Column": column,
                "Missing Count": int(missing_count),
                "Missing Percentage": round(missing_percentage, 2),
                "Data Type": str(df[column].dtype)
            })

missing_details_df = pd.DataFrame(missing_value_details)

if not missing_details_df.empty:
    display(
        missing_details_df.sort_values(
            by="Missing Count",
            ascending=False
        )
    )
else:
    print("No missing values found in any dataset.")

,Dataset,Column,Missing Count,Missing Percentage,Data Type
0,essential_skills,Not Relevant,9100,50.0,object


In [22]:
# ============================================================
# INSPECT ROWS CONTAINING MISSING VALUES
# ============================================================

for dataset_name, df in raw_data.items():
    
    missing_columns = df.columns[df.isnull().any()].tolist()
    
    if missing_columns:
        
        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_name.upper()}")
        print("COLUMNS WITH MISSING VALUES:")
        print(missing_columns)
        print("=" * 80)
        
        display(
            df[df[missing_columns].isnull().any(axis=1)]
            .head(10)
        )


DATASET: ESSENTIAL_SKILLS
COLUMNS WITH MISSING VALUES:
['Not Relevant']


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.0000,4.0000,4.0000,N,NaN,08/2023,Analyst
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst
6,11-1011.00,Chief Executives,2.A.1.d,Speaking,IM,Importance,4.25,8,0.1637,3.9292,4.5708,N,NaN,08/2023,Analyst
8,11-1011.00,Chief Executives,2.A.1.e,Mathematics,IM,Importance,3.25,8,0.2500,2.7600,3.7400,N,NaN,08/2023,Analyst
10,11-1011.00,Chief Executives,2.A.1.f,Science,IM,Importance,1.62,8,0.1830,1.2664,1.9836,N,NaN,08/2023,Analyst
12,11-1011.00,Chief Executives,2.A.2.a,Critical Thinking,IM,Importance,4.38,8,0.1830,4.0164,4.7336,N,NaN,08/2023,Analyst
14,11-1011.00,Chief Executives,2.A.2.b,Active Learning,IM,Importance,3.75,8,0.1637,3.4292,4.0708,N,NaN,08/2023,Analyst
16,11-1011.00,Chief Executives,2.A.2.c,Learning Strategies,IM,Importance,3.12,8,0.1250,2.8800,3.3700,N,NaN,08/2023,Analyst
18,11-1011.00,Chief Executives,2.A.2.d,Monitoring,IM,Importance,4.00,8,0.0000,4.0000,4.0000,N,NaN,08/2023,Analyst


In [23]:
# ============================================================
# CREATE CLEAN COPIES OF ALL DATASETS
# ============================================================

cleaned_data = {}

for name, df in raw_data.items():
    
    # Create a copy so original raw_data remains unchanged
    cleaned_df = df.copy()
    
    cleaned_data[name] = cleaned_df

print("Clean copies created successfully.")

for name, df in cleaned_data.items():
    print(f"{name}: {df.shape}")

Clean copies created successfully.
employee_attrition: (1470, 35)
performance_engagement: (5000, 13)
occupational_data: (1016, 3)
essential_skills: (18200, 15)
software_skills: (31821, 7)


In [24]:
# ============================================================
# HANDLE MISSING VALUES
# ============================================================

# Fill missing values in essential_skills
if "essential_skills" in cleaned_data:
    
    essential_skills = cleaned_data["essential_skills"]
    
    if "Not Relevant" in essential_skills.columns:
        
        missing_before = essential_skills["Not Relevant"].isnull().sum()
        
        essential_skills["Not Relevant"] = (
            essential_skills["Not Relevant"]
            .fillna("N")
        )
        
        missing_after = essential_skills["Not Relevant"].isnull().sum()
        
        print("Essential Skills - 'Not Relevant' column")
        print(f"Missing values before: {missing_before}")
        print(f"Missing values after: {missing_after}")

Essential Skills - 'Not Relevant' column
Missing values before: 9100
Missing values after: 0


In [25]:
# ============================================================
# REMOVE DUPLICATE ROWS
# ============================================================

duplicate_report = []

for name, df in cleaned_data.items():
    
    duplicates_before = df.duplicated().sum()
    
    df.drop_duplicates(inplace=True)
    
    duplicates_after = df.duplicated().sum()
    
    duplicate_report.append({
        "Dataset": name,
        "Duplicates Before": duplicates_before,
        "Duplicates After": duplicates_after,
        "Rows After Cleaning": df.shape[0]
    })

duplicate_report_df = pd.DataFrame(duplicate_report)

duplicate_report_df

,Dataset,Duplicates Before,Duplicates After,Rows After Cleaning
0,employee_attrition,0,0,1470
1,performance_engagement,0,0,5000
2,occupational_data,0,0,1016
3,essential_skills,0,0,18200
4,software_skills,0,0,31821


In [26]:
# ============================================================
# STANDARDIZE TEXT DATA
# ============================================================

for name, df in cleaned_data.items():
    
    for column in df.select_dtypes(include="object").columns:
        
        # Remove unnecessary spaces
        df[column] = df[column].astype(str).str.strip()
        
        # Convert empty strings to NaN
        df[column] = df[column].replace("", pd.NA)

print("Text columns standardized successfully.")

Text columns standardized successfully.


In [27]:
# ============================================================
# FINAL DATA VALIDATION
# ============================================================

final_validation = []

for name, df in cleaned_data.items():
    
    total_missing = df.isnull().sum().sum()
    duplicate_rows = df.duplicated().sum()
    
    if total_missing == 0 and duplicate_rows == 0:
        status = "PASS"
    else:
        status = "REVIEW"
    
    final_validation.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": total_missing,
        "Duplicate Rows": duplicate_rows,
        "Validation Status": status
    })

final_validation_report = pd.DataFrame(final_validation)

final_validation_report

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Validation Status
0,employee_attrition,1470,35,0,0,PASS
1,performance_engagement,5000,13,0,0,PASS
2,occupational_data,1016,3,0,0,PASS
3,essential_skills,18200,15,0,0,PASS
4,software_skills,31821,7,0,0,PASS


In [28]:
# ============================================================
# SAVE CLEANED DATASETS TO PROCESSED FOLDER
# ============================================================

saved_files = []

for name, df in cleaned_data.items():
    
    output_path = PROCESSED_DATA_PATH / f"{name}_cleaned.csv"
    
    df.to_csv(output_path, index=False)
    
    saved_files.append({
        "Dataset": name,
        "File Path": str(output_path),
        "Rows Saved": df.shape[0]
    })
    
    print(f"✓ Saved: {output_path.name}")

saved_files_df = pd.DataFrame(saved_files)

print("\nAll cleaned datasets saved successfully!\n")

saved_files_df

✓ Saved: employee_attrition_cleaned.csv
✓ Saved: performance_engagement_cleaned.csv
✓ Saved: occupational_data_cleaned.csv
✓ Saved: essential_skills_cleaned.csv
✓ Saved: software_skills_cleaned.csv

All cleaned datasets saved successfully!



,Dataset,File Path,Rows Saved
0,employee_attrition,C:\Users\Shubham\Desktop\enterprise_hr_ai\data...,1470
1,performance_engagement,C:\Users\Shubham\Desktop\enterprise_hr_ai\data...,5000
2,occupational_data,C:\Users\Shubham\Desktop\enterprise_hr_ai\data...,1016
3,essential_skills,C:\Users\Shubham\Desktop\enterprise_hr_ai\data...,18200
4,software_skills,C:\Users\Shubham\Desktop\enterprise_hr_ai\data...,31821


In [29]:
# ============================================================
# VERIFY PROCESSED FILES
# ============================================================

print("Processed Data Directory:")
print(PROCESSED_DATA_PATH)

print("\nFiles saved:\n")

for file in PROCESSED_DATA_PATH.glob("*.csv"):
    
    file_size_kb = file.stat().st_size / 1024
    
    print(
        f"✓ {file.name} "
        f"({file_size_kb:.2f} KB)"
    )

Processed Data Directory:
C:\Users\Shubham\Desktop\enterprise_hr_ai\data\processed

Files saved:

✓ employee_attrition_cleaned.csv (222.63 KB)
✓ essential_skills_cleaned.csv (2208.82 KB)
✓ occupational_data_cleaned.csv (261.75 KB)
✓ performance_engagement_cleaned.csv (398.83 KB)
✓ software_skills_cleaned.csv (3474.17 KB)


In [30]:
# ============================================================
# DATA CLEANING SUMMARY
# ============================================================

data_cleaning_summary = pd.DataFrame({
    "Dataset": list(cleaned_data.keys()),
    "Rows": [df.shape[0] for df in cleaned_data.values()],
    "Columns": [df.shape[1] for df in cleaned_data.values()],
    "Missing Values": [
        df.isnull().sum().sum()
        for df in cleaned_data.values()
    ],
    "Duplicate Rows": [
        df.duplicated().sum()
        for df in cleaned_data.values()
    ],
    "Status": ["READY FOR ANALYSIS"] * len(cleaned_data)
})

data_cleaning_summary

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Status
0,employee_attrition,1470,35,0,0,READY FOR ANALYSIS
1,performance_engagement,5000,13,0,0,READY FOR ANALYSIS
2,occupational_data,1016,3,0,0,READY FOR ANALYSIS
3,essential_skills,18200,15,0,0,READY FOR ANALYSIS
4,software_skills,31821,7,0,0,READY FOR ANALYSIS
